In [1]:
import os

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import torch

In [2]:
from os import path

args = {}

args["project_root"] = path.join(path.abspath(""), "..")
args["captures_dir"] = path.join(args["project_root"],
                                 "dataset", "captures")
args["references_dir"] = path.join(args["project_root"],
                                   "dataset", "references")
args["detections_dir"] = path.join(args["project_root"],
                                   "dataset", "detections")
args["web_faces_dir"] = path.join(args["project_root"],
                                  "dataset", "web_faces")

args["cr_imsize"] = 160
args["imwidth"] = 800
args["imheight"] = 600
args["imchannels"] = 3

# args["device"] = torch.device("cuda"if torch.cuda.is_available()
#                               else "cpu")
args["device"] = torch.device("cpu")
args["dtype"] = torch.float32

In [3]:
"""
Create dataset and data loader for the reference image database.
"""

from torchvision import datasets
from torch.utils.data import DataLoader

reference_set = datasets.ImageFolder(args["references_dir"])
reference_set.idx_to_class = {i: c for c, i in
                              reference_set.class_to_idx.items()}


def collate_fn(batch):
    data, targets = [], []

    for d, t in batch:
        data.append(d.resize((args["imwidth"], args["imheight"])))
        targets.append(t)
        
    return data, targets


reference_loader = DataLoader(reference_set, batch_size=8,
                              collate_fn=collate_fn, shuffle=True)

In [5]:
"""
Detect cropped faces using MTCNN imp in facenet_pytorch
"""

from facenet_pytorch import MTCNN

mtcnn = MTCNN(
    image_size=args["cr_imsize"],
    keep_all=False,
    margin=0,
    post_process=False,
    select_largest=True)

In [6]:
cropped_ref_faces = []
ref_targs = []

for images, labels in reference_loader:
    cropped_ref_faces.extend(mtcnn(images))
    ref_targs.extend(labels)

/home/prixa-ml/anaconda3/lib/python3.8/site-packages/torch/nn/functional.py:718: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at  /opt/conda/conda-bld/pytorch_1623448278899/work/c10/core/TensorImpl.h:1156.)
  return torch.max_pool2d(input, kernel_size, stride, padding, dilation, ceil_mode)


In [7]:
"""
Save the detected faces to disk in pytorch supported format
"""

import os
from os import path
import matplotlib.pyplot as plt


def save_labelled_images(images, labels, img_root, ext="jpg"):
    # Write obtained images to disk

    try:
        img_root = path.abspath(img_root)
        os.stat(img_root)
    except FileNotFoundError:
        os.mkdir(img_root)

    lb_to_fname = {}

    for img, label in zip(images, labels):
        label = str(label)
        img_dir = path.join(img_root, label)

        if label not in lb_to_fname:
            lb_to_fname[label] = 1
        else:
            lb_to_fname[label] += 1

        try:
            os.stat(img_dir)
        except FileNotFoundError:
            os.mkdir(img_dir)

        plt.imsave(
            path.join(
                img_dir, "{fname:0>3}.{ext}".format(
                    fname=lb_to_fname[label],
                    ext=ext)),
            img)


def _prepare_img_for_save(img):
    if img is None:
        return np.zeros((args["cr_imsize"],
                         args["cr_imsize"],
                         args["imchannels"]), dtype="uint8")
    else:
        return np.array(img.permute((1, 2, 0)), dtype="uint8")


save_labelled_images(map(_prepare_img_for_save, cropped_ref_faces),
                     map(lambda l: reference_set.idx_to_class[l],
                         ref_targs),
                     args["detections_dir"])

In [8]:
"""
Load cropped faces from disk
"""

from torchvision import datasets

cropped_faces_set = datasets.ImageFolder(args["detections_dir"])
cropped_faces_set.idx_to_class = {i: c for c, i in
                                  cropped_faces_set.class_to_idx.items()}


def collate_fn(batch):
    data, labels = [], []

    for d, l in batch:
        # Resize the image and clip to
        # the specified number of image channels
        resized = np.array(
            d.resize((args["cr_imsize"],
                      args["cr_imsize"]))
        ).transpose((2, 0, 1))[:args["imchannels"]]
        
        data.append(
            torch.tensor(resized, dtype=args["dtype"],
                         device=args["device"]))
        labels.append(l)

    return torch.stack(data), torch.tensor(labels,
                                           dtype=args["dtype"],
                                           device=args["device"])


cropped_faces_loader = DataLoader(cropped_faces_set, batch_size=8,
                                  collate_fn=collate_fn, shuffle=True)

In [9]:
"""
Perform fixed image standardization on the loaded images and feed them into InceptionResnetV1
"""

from facenet_pytorch import InceptionResnetV1, fixed_image_standardization

resnet = InceptionResnetV1(
    pretrained="vggface2",
    classify=False).eval().to(device=args["device"])

In [10]:
reference_embeddings = []
reference_targets = []

for images, targets in cropped_faces_loader:
    images = fixed_image_standardization(images)
    reference_embeddings.extend(resnet(images).detach().cpu())
    reference_targets.extend(targets.detach().cpu())
    
reference_embeddings = torch.stack(reference_embeddings)
reference_targets = torch.stack(reference_targets)

In [11]:
len(reference_embeddings), reference_embeddings[0].shape

(103, torch.Size([512]))

In [15]:
"""
Capture image and commence live detection
"""

from video_utils import vid_cam

CAMERA_DEVICE = 0
LIVE_CAPTURE_NAME = "live_capture.jpg"

vid_cam.capture_image_from_stream(CAMERA_DEVICE,
                                  imgdir=args["captures_dir"],
                                  imgname=LIVE_CAPTURE_NAME)

QObject::moveToThread: Current thread (0x55a926d55c10) is not the object's thread (0x55a926ae0b10).
Cannot move to target thread (0x55a926d55c10)

QObject::moveToThread: Current thread (0x55a926d55c10) is not the object's thread (0x55a926ae0b10).
Cannot move to target thread (0x55a926d55c10)

QObject::moveToThread: Current thread (0x55a926d55c10) is not the object's thread (0x55a926ae0b10).
Cannot move to target thread (0x55a926d55c10)

QObject::moveToThread: Current thread (0x55a926d55c10) is not the object's thread (0x55a926ae0b10).
Cannot move to target thread (0x55a926d55c10)

QObject::moveToThread: Current thread (0x55a926d55c10) is not the object's thread (0x55a926ae0b10).
Cannot move to target thread (0x55a926d55c10)

QObject::moveToThread: Current thread (0x55a926d55c10) is not the object's thread (0x55a926ae0b10).
Cannot move to target thread (0x55a926d55c10)

QObject::moveToThread: Current thread (0x55a926d55c10) is not the object's thread (0x55a926ae0b10).
Cannot move to tar

In [16]:
from os import path
from PIL import Image

live_image = Image.open(path.join(
    args["captures_dir"], LIVE_CAPTURE_NAME)
).resize((args["imwidth"], args["imheight"]))

cropped_live_imgs = []
cropped_live_imgs.append(mtcnn(live_image))
cropped_live_imgs = torch.stack(cropped_live_imgs)

cropped_live_imgs = fixed_image_standardization(cropped_live_imgs)
live_imgs_embeddings = resnet(cropped_live_imgs).detach().cpu()

In [17]:
live_imgs_embeddings.shape

torch.Size([1, 512])

In [18]:
import pandas as pd

euc_dist = [(e1 - e2).norm()
                  for e1 in live_imgs_embeddings
                  for e2 in reference_embeddings]
cos_sim = [torch.dot(e1, e2) /
                 (e1.norm() * e2.norm())
                 for e1 in live_imgs_embeddings
                 for e2 in reference_embeddings]

display(pd.DataFrame(euc_dist, columns=["Euc Dist"]))
display(pd.DataFrame(cos_sim, columns=["Cos Sim"]))

,Euc Dist
0,1.163947
1,1.417533
2,1.280170
3,1.246184
4,1.354629
...,...
98,1.369585
99,1.514445
100,1.332192
101,1.410692


,Cos Sim
0,0.322614
1,-0.004700
2,0.180582
3,0.223513
4,0.082490
...,...
98,0.062118
99,-0.146772
100,0.112632
101,0.004974


In [19]:
e_pred_targ = int(reference_targets[np.argmin(euc_dist)].item())
c_pred_targ = int(reference_targets[np.argmax(cos_sim)].item())
print(cropped_faces_set.idx_to_class[e_pred_targ],
      cropped_faces_set.idx_to_class[c_pred_targ])

RaviPrajapati RaviPrajapati
